# CSE Final Year Capstone Project — RAG & Open-Source LLMs
## Building a Retrieval-Augmented Generation (RAG) Knowledge Assistant
**Department of Computer Science & Engineering**  
**Duration:** 4 Weeks | **Team Project:** Knowledge Assistant  
---
This Jupyter/Google Colab Notebook contains the complete, step-by-step implementation for the B.Tech Capstone Project following all 6 stages of Retrieval-Augmented Generation.

### 3.1 Setup
Install required dependencies: LlamaIndex, ChromaDB, HuggingFace Embeddings, Groq LLM integration, and Streamlit.

In [ ]:
!pip install -q llama-index llama-index-embeddings-huggingface \
 llama-index-llms-groq chromadb llama-index-vector-stores-chroma \
 streamlit pypdf groq

### 3.2 Loading Your Documents
Load source documents (PDF, TXT, MD) using LlamaIndex's `SimpleDirectoryReader` to preserve text and metadata.

In [ ]:
import os
from llama_index.core import SimpleDirectoryReader

# Set document directory path (e.g. data/raw or data/)
data_dir = "data/raw" if os.path.exists("data/raw") else "data"
documents = SimpleDirectoryReader(data_dir, recursive=True).load_data()

print(f"Loaded {len(documents)} document section(s)")
if documents:
    print("--- Sample Document Preview ---")
    print(documents[0].text[:500])
    print("Metadata:", documents[0].metadata)

### 3.3 Chunking the Documents
Split large text documents into coherent, overlapping chunks using `SentenceSplitter` with `chunk_size=512` and `chunk_overlap=64`.

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(
    chunk_size=512,  # max tokens per chunk
    chunk_overlap=64  # tokens shared between consecutive chunks
)

nodes = splitter.get_nodes_from_documents(documents)
print(f"Created {len(nodes)} chunks from {len(documents)} document sections")
if nodes:
    print("--- Sample Chunk Preview ---")
    print(nodes[0].text[:300])

### 3.4 Generating Embeddings
Convert each chunk into a 384-dimensional dense vector using `sentence-transformers/all-MiniLM-L6-v2`.

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Sanity check embedding generation
test_vector = embed_model.get_text_embedding("What is retrieval augmented generation?")
print(f"Embedding length: {len(test_vector)}")  # Should print 384

### 3.5 Building the Vector Index with ChromaDB
Persist vectors into ChromaDB vector database and construct `VectorStoreIndex`.

In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex, StorageContext, Settings

# Configure global embedding model
Settings.embed_model = embed_model

# Initialize ChromaDB Persistent Client
chroma_client = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = chroma_client.get_or_create_collection("capstone_knowledge_base")

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex(nodes, storage_context=storage_context)

print("Vector index built and saved to ./chroma_db")

# Test Similarity Retrieval in Isolation
retriever = index.as_retriever(similarity_top_k=3)
results = retriever.retrieve("How can an NGO register in India?")
print("\n--- RETRIEVAL SANITY TEST ---")
for r in results:
    file_name = r.node.metadata.get('file_name', 'unknown')
    print(f"Score: {r.score:.3f} | Source: {file_name}")
    print(r.node.text[:200], "\n---")

### 3.6 Connecting an Open-Source LLM (Groq API)
Connect to high-speed Llama-3.1-8B model via Groq API.

In [ ]:
import os
from llama_index.llms.groq import Groq

# Get API Key from environment or Google Colab secrets
try:
    from google.colab import userdata
    groq_key = userdata.get("GROQ_API_KEY")
except Exception:
    groq_key = os.getenv("GROQ_API_KEY", "YOUR_GROQ_API_KEY_HERE")

if groq_key and groq_key != "YOUR_GROQ_API_KEY_HERE":
    llm = Groq(model="llama-3.1-8b-instant", api_key=groq_key)
    Settings.llm = llm
    response = llm.complete("In one sentence, what is Retrieval-Augmented Generation?")
    print("LLM Test Response:", response)
else:
    print("Please set your GROQ_API_KEY in environment variables or Google Colab Secrets.")

### 3.7 Complete RAG Query Engine & Grounded Prompting
Enforce strict source grounding to prevent hallucinations using a custom QA Prompt Template.

In [ ]:
from llama_index.core import PromptTemplate

qa_template = PromptTemplate(
    "You are a helpful assistant that answers ONLY using the context below.\n"
    "If the answer is not contained in the context, say "
    "'I don't have enough information to answer that.'\n\n"
    "Context:\n{context_str}\n\n"
    "Question: {query_str}\n"
    "Answer:"
)

query_engine = index.as_query_engine(
    similarity_top_k=3,
    text_qa_template=qa_template
)

# Test query
test_query = "What are the legal options for NGO registration in India?"
response = query_engine.query(test_query)
print("ANSWER:\n", response)
print("\nSOURCES USED:")
for node in response.source_nodes:
    print("-", node.metadata.get("file_name", "Unknown"), f"(score: {node.score:.3f})")

### 3.8 Streamlit Chat Application Code
Below code generates the `app_capstone.py` file for launching the Streamlit interface.

In [ ]:
streamlit_code = '''import streamlit as st
from llama_index.core import VectorStoreIndex, Settings, StorageContext, PromptTemplate
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb, os

st.set_page_config(page_title="Team RAG Knowledge Assistant", page_icon="📚")
st.title("📚 Team RAG Knowledge Assistant")

@st.cache_resource
def load_query_engine():
    Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
    api_key = os.environ.get("GROQ_API_KEY", "")
    if api_key:
        Settings.llm = Groq(model="llama-3.1-8b-instant", api_key=api_key)
    client = chromadb.PersistentClient(path="./chroma_db")
    collection = client.get_or_create_collection("capstone_knowledge_base")
    vector_store = ChromaVectorStore(chroma_collection=collection)
    index = VectorStoreIndex.from_vector_store(vector_store)
    
    qa_template = PromptTemplate(
        "You are a helpful assistant that answers ONLY using the context below.\
"
        "If the answer is not contained in the context, say \
"
        "'I don\'t have enough information to answer that.'\
\
"
        "Context:\
{context_str}\
\
"
        "Question: {query_str}\
"
        "Answer:"
    )
    return index.as_query_engine(similarity_top_k=3, text_qa_template=qa_template)

try:
    query_engine = load_query_engine()
except Exception as e:
    st.error(f"Error initializing query engine: {e}")
    st.stop()

if "history" not in st.session_state:
    st.session_state.history = []

question = st.chat_input("Ask a question about the knowledge base...")
if question:
    response = query_engine.query(question)
    sources = sorted({n.metadata.get("file_name", "unknown") for n in response.source_nodes})
    st.session_state.history.append((question, str(response), sources))

for q, a, srcs in st.session_state.history:
    with st.chat_message("user"):
        st.write(q)
    with st.chat_message("assistant"):
        st.write(a)
        st.caption("Sources: " + ", ".join(srcs))
'''

with open("app_capstone.py", "w") as f:
    f.write(streamlit_code)
print("Saved app_capstone.py successfully!")

### 4.1 Automated 15-Question Evaluation Suite
Run benchmark evaluation across 15 questions to check retrieval accuracy and grounded fallback behavior.

In [ ]:
import json

test_questions_path = "evaluation/test_questions.json"
if os.path.exists(test_questions_path):
    with open(test_questions_path, "r") as f:
        questions = json.load(f)
    
    print(f"--- Running Evaluation on {len(questions)} Questions ---")
    for i, item in enumerate(questions, 1):
        q = item["question"]
        print(f"\n[Q{i:02d}] {q}")
        try:
            res = query_engine.query(q)
            print(f"Answer: {str(res)[:200]}...")
            sources = [n.metadata.get('file_name', 'unknown') for n in res.source_nodes]
            print(f"Sources: {sources}")
        except Exception as err:
            print(f"Execution Note: {err}")